# SVM Lab: Classification and Regression on `cyber_15000.csv`

This lab uses the attached cyber-security dataset to practice **Support Vector Machine (SVM)** methods for both:

- **Multiclass classification** using `attack_type`
- **Regression** using `incident_impact_score`

## Learning goals
By the end of this lab, you should be able to:

1. inspect a real dataset before modeling,
2. preprocess mixed cyber-security features for SVM,
3. train and evaluate **Linear SVM** and **RBF SVM** classifiers,
4. train and evaluate **Linear SVR** and **RBF SVR** regressors,
5. compare models using appropriate metrics, and
6. perform a small hyperparameter search.

> In every section below, the **task** is written first, followed by a **solution approach** in the markdown cell before the code.


In [ ]:

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC, LinearSVR, SVR
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    r2_score,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

DATA_CANDIDATES = [
    "cyber_15000.csv",
    "/mnt/data/cyber_15000.csv",
]

for candidate in DATA_CANDIDATES:
    if os.path.exists(candidate):
        DATA_PATH = candidate
        break
else:
    raise FileNotFoundError("Could not locate cyber_15000.csv. Put the dataset next to this notebook or update DATA_PATH.")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

print("Using dataset:", DATA_PATH)



## Task 1 — Load the dataset and inspect its structure

**Task.** Load the CSV file, display the first rows, check the shape of the dataset, and inspect the target columns for classification and regression.

**Solution approach.** We first read the dataset with pandas, then inspect its size, columns, target distribution, and the descriptive statistics of the regression target.


In [ ]:

df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
print("\nFirst 5 rows:")
display(df_raw.head())

print("\nClassification target distribution:")
display(df_raw["attack_type"].value_counts())

print("\nRegression target summary:")
display(df_raw["incident_impact_score"].describe())



## Task 2 — Preprocess the dataset for SVM

**Task.** Prepare the dataset for SVM learning. Convert the timestamp into useful numerical features, drop obvious identifier columns, and make sure boolean columns are numeric.

**Solution approach.** SVM models require numerical input. We convert `timestamp_utc` into `hour`, `dayofweek`, and `is_weekend`, remove `record_id`, and convert boolean columns to integers.


In [ ]:

def prepare_dataset(df):
    df = df.copy()
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])
    df["hour"] = df["timestamp_utc"].dt.hour
    df["dayofweek"] = df["timestamp_utc"].dt.dayofweek
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

    df = df.drop(columns=["timestamp_utc", "record_id"])

    bool_cols = df.select_dtypes(include="bool").columns
    for col in bool_cols:
        df[col] = df[col].astype(int)

    return df

df = prepare_dataset(df_raw)

print("Processed shape:", df.shape)
print("Boolean columns converted:", len(df.select_dtypes(include='int').columns))
print("\nSample of processed columns:")
display(df.head())



## Task 3 — Build an SVM classification dataset

**Task.** Use the processed features to predict the multiclass target `attack_type`. Split the data into training and test sets.

**Solution approach.** We remove the target columns from the feature matrix, keep `attack_type` as the classification label, and use a **stratified** split to preserve class proportions.


In [ ]:

X_cls = df.drop(columns=["attack_type", "incident_impact_score"])
y_cls = df["attack_type"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls,
    y_cls,
    test_size=0.20,
    random_state=42,
    stratify=y_cls,
)

print("Classification train shape:", Xc_train.shape)
print("Classification test shape :", Xc_test.shape)
print("\nClass distribution in training set:")
display(yc_train.value_counts(normalize=True).sort_index().round(3))



## Task 4 — Train a baseline Linear SVM classifier

**Task.** Train a baseline **Linear SVM** for multiclass classification.

**Solution approach.** Because the dataset is large and high-dimensional, `LinearSVC` is a practical baseline. We use a pipeline with `StandardScaler` followed by `LinearSVC`.


In [ ]:

linear_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearSVC(random_state=42, dual=False, max_iter=5000)),
])

linear_svm_clf.fit(Xc_train, yc_train)
yc_pred_linear = linear_svm_clf.predict(Xc_test)

linear_acc = accuracy_score(yc_test, yc_pred_linear)
linear_f1 = f1_score(yc_test, yc_pred_linear, average="weighted")

print("Linear SVM accuracy     :", round(linear_acc, 4))
print("Linear SVM weighted F1  :", round(linear_f1, 4))



## Task 5 — Evaluate the classification model

**Task.** Evaluate the Linear SVM using a classification report and a confusion matrix.

**Solution approach.** For multiclass and somewhat imbalanced data, **weighted F1-score** is more informative than accuracy alone. The confusion matrix helps reveal which attack categories are confused most often.


In [ ]:

print(classification_report(yc_test, yc_pred_linear))

labels = sorted(y_cls.unique())
cm = confusion_matrix(yc_test, yc_pred_linear, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix - Linear SVM")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()



## Task 6 — Train an RBF SVM classifier on a balanced sample

**Task.** Train an **RBF-kernel SVM** classifier and compare it with the linear model.

**Solution approach.** Kernel SVM can be computationally expensive on a large dataset, so we create a **balanced sample** with the same number of instances per class. This makes the experiment faster and fairer for comparison.


In [ ]:

balanced_cls_df = (
    df.groupby("attack_type", group_keys=False)
      .apply(lambda g: g.sample(min(len(g), 400), random_state=42))
      .reset_index(drop=True)
)

X_cls_small = balanced_cls_df.drop(columns=["attack_type", "incident_impact_score"])
y_cls_small = balanced_cls_df["attack_type"]

Xcs_train, Xcs_test, ycs_train, ycs_test = train_test_split(
    X_cls_small,
    y_cls_small,
    test_size=0.20,
    random_state=42,
    stratify=y_cls_small,
)

rbf_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", C=3, gamma="scale", random_state=42)),
])

rbf_svm_clf.fit(Xcs_train, ycs_train)
ycs_pred_rbf = rbf_svm_clf.predict(Xcs_test)

rbf_acc = accuracy_score(ycs_test, ycs_pred_rbf)
rbf_f1 = f1_score(ycs_test, ycs_pred_rbf, average="weighted")

comparison_cls = pd.DataFrame({
    "Model": ["Linear SVM (full data)", "RBF SVM (balanced sample)"],
    "Accuracy": [linear_acc, rbf_acc],
    "Weighted F1": [linear_f1, rbf_f1],
})

display(comparison_cls.round(4))



## Task 7 — Tune the classification SVM with Grid Search

**Task.** Perform a small hyperparameter search for the RBF classifier.

**Solution approach.** To keep the tuning manageable, we use a smaller balanced subset and a small search grid over `C` and `gamma`. The scoring metric is **weighted F1**.


In [ ]:

tiny_cls_df = (
    df.groupby("attack_type", group_keys=False)
      .apply(lambda g: g.sample(min(len(g), 100), random_state=42))
      .reset_index(drop=True)
)

X_cls_tiny = tiny_cls_df.drop(columns=["attack_type", "incident_impact_score"])
y_cls_tiny = tiny_cls_df["attack_type"]

cls_grid = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", random_state=42)),
    ]),
    param_grid={
        "model__C": [1, 3],
        "model__gamma": ["scale", 0.01],
    },
    scoring="f1_weighted",
    cv=2,
    n_jobs=1,
)

cls_grid.fit(X_cls_tiny, y_cls_tiny)

print("Best classification parameters:", cls_grid.best_params_)
print("Best weighted F1 from CV      :", round(cls_grid.best_score_, 4))



## Task 8 — Prepare the regression dataset

**Task.** Use the same processed data to predict the continuous target `incident_impact_score`.

**Solution approach.** For regression, we drop `incident_impact_score` from the input features and keep it as the target variable. We also remove `attack_type` to avoid using the class label as a predictor.


In [ ]:

X_reg = df.drop(columns=["incident_impact_score", "attack_type"])
y_reg = df["incident_impact_score"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42,
)

print("Regression train shape:", Xr_train.shape)
print("Regression test shape :", Xr_test.shape)



## Task 9 — Train a baseline Linear SVR model

**Task.** Train a **Linear SVR** model to predict `incident_impact_score`.

**Solution approach.** `LinearSVR` is a good baseline for high-dimensional regression. We scale the features first because SVM-based models are sensitive to feature magnitudes.


In [ ]:

linear_svr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearSVR(random_state=42, max_iter=5000)),
])

linear_svr.fit(Xr_train, yr_train)
yr_pred_linear = linear_svr.predict(Xr_test)

linear_mae = mean_absolute_error(yr_test, yr_pred_linear)
linear_rmse = rmse(yr_test, yr_pred_linear)
linear_r2 = r2_score(yr_test, yr_pred_linear)

print("Linear SVR MAE  :", round(linear_mae, 4))
print("Linear SVR RMSE :", round(linear_rmse, 4))
print("Linear SVR R^2  :", round(linear_r2, 4))



## Task 10 — Evaluate the regression model

**Task.** Evaluate the Linear SVR using MAE, RMSE, and \(R^2\). Also visualize residuals.

**Solution approach.** Residual analysis helps us see whether the model makes mostly small errors or whether there are systematic mistakes at some prediction ranges.


In [ ]:

residuals = yr_test - yr_pred_linear

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(yr_pred_linear, residuals, alpha=0.35)
ax.axhline(0, linestyle="--")
ax.set_xlabel("Predicted impact score")
ax.set_ylabel("Residual (true - predicted)")
ax.set_title("Residual Plot - Linear SVR")
plt.tight_layout()
plt.show()



## Task 11 — Train an RBF SVR model on a sample

**Task.** Train an **RBF-kernel SVR** regressor and compare it with the linear model.

**Solution approach.** Kernel SVR can be slow on a large dataset, so we train it on a random sample. We then compare the regression metrics against the linear baseline.


In [ ]:

reg_sample_df = df.sample(3000, random_state=42)

X_reg_small = reg_sample_df.drop(columns=["incident_impact_score", "attack_type"])
y_reg_small = reg_sample_df["incident_impact_score"]

Xrs_train, Xrs_test, yrs_train, yrs_test = train_test_split(
    X_reg_small,
    y_reg_small,
    test_size=0.20,
    random_state=42,
)

rbf_svr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR(kernel="rbf", C=10, epsilon=0.2, gamma="scale")),
])

rbf_svr.fit(Xrs_train, yrs_train)
yrs_pred_rbf = rbf_svr.predict(Xrs_test)

rbf_mae = mean_absolute_error(yrs_test, yrs_pred_rbf)
rbf_rmse = rmse(yrs_test, yrs_pred_rbf)
rbf_r2 = r2_score(yrs_test, yrs_pred_rbf)

comparison_reg = pd.DataFrame({
    "Model": ["Linear SVR (full data)", "RBF SVR (sample of 3000)"],
    "MAE": [linear_mae, rbf_mae],
    "RMSE": [linear_rmse, rbf_rmse],
    "R^2": [linear_r2, rbf_r2],
})

display(comparison_reg.round(4))



## Task 12 — Tune the regression SVM with Grid Search

**Task.** Perform a small hyperparameter search for the SVR model.

**Solution approach.** We use a smaller sample and search over a small set of `C` and `epsilon` values. The scoring metric is negative MAE, so larger (less negative) is better.


In [ ]:

tiny_reg_df = df.sample(1200, random_state=42)

X_reg_tiny = tiny_reg_df.drop(columns=["incident_impact_score", "attack_type"])
y_reg_tiny = tiny_reg_df["incident_impact_score"]

reg_grid = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf")),
    ]),
    param_grid={
        "model__C": [1, 10],
        "model__epsilon": [0.1, 0.5],
    },
    scoring="neg_mean_absolute_error",
    cv=2,
    n_jobs=1,
)

reg_grid.fit(X_reg_tiny, y_reg_tiny)

print("Best regression parameters:", reg_grid.best_params_)
print("Best CV score (negative MAE):", round(reg_grid.best_score_, 4))



## Task 13 — Final discussion

**Task.** Summarize what you learned from the SVM experiments.

**Solution approach.** The cell below produces a compact summary table. In your own write-up, comment on:

1. why scaling is important for SVM,
2. why Linear SVM is practical for high-dimensional data,
3. why kernel SVM often needs sampling or careful tuning,
4. which metrics are best for classification and regression here, and
5. whether the data appears linearly separable for all classes.


In [ ]:

final_summary = pd.DataFrame({
    "Experiment": [
        "Linear SVM classification",
        "RBF SVM classification",
        "Linear SVR regression",
        "RBF SVR regression",
    ],
    "Main metric": [
        f"Weighted F1 = {linear_f1:.4f}",
        f"Weighted F1 = {rbf_f1:.4f}",
        f"RMSE = {linear_rmse:.4f}",
        f"RMSE = {rbf_rmse:.4f}",
    ],
})

display(final_summary)
